# Opus-MT + LoRA: Classical Chinese Poetry → English

Fine-tunes `Helsinki-NLP/opus-mt-zh-en` (MarianMT, ~74M params) on the PoetMT poetry dataset using LoRA.

**Why opus-mt over mT5-base?** opus-mt-zh-en is already pre-trained on OPUS Chinese→English data, so it starts as a working translator. mT5-base was only pre-trained on unsupervised text and cannot translate without extensive fine-tuning first.

**Estimated training time (15 epochs):**
- Free Colab T4: ~20–35 min
- Local GPU (RTX 3060-class): ~50–70 min

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers peft accelerate sentencepiece sacrebleu evaluate datasets
!pip install -q "torchao>=0.16.0"


In [ ]:
# ── Cell 2: Mount Google Drive (adapter will be saved here) ───────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 3: Clone repo (private) ─────────────────────────────────────────
# Colab sidebar → Secrets → add GITHUB_TOKEN (fine-grained PAT, contents read)
import os
from google.colab import userdata
from getpass import getpass

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None
if not GITHUB_TOKEN:
    GITHUB_TOKEN = getpass("Paste GitHub PAT: ")

REPO_DIR = "/content/chinese_poetry_translation"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/emmah-3815/chinese_poetry_translation.git"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git checkout juqy-dev
!git pull
print("Working dir:", os.getcwd())
!git log --oneline -3


In [ ]:
# ── Cell 4: Download raw data (PoetMT + CCPM) ────────────────────────────
%cd /content/chinese_poetry_translation
import os

# Clone PoetMT into data/PoetMT-main/PoetMT-main so all_poems/ is at the expected path
if not os.path.exists("data/PoetMT-main/PoetMT-main/all_poems/tang.jsonl"):
    os.makedirs("data/PoetMT-main", exist_ok=True)
    !git clone https://github.com/andongBlue/PoetMT.git data/PoetMT-main/PoetMT-main
    print("PoetMT contents:", os.listdir("data/PoetMT-main/PoetMT-main"))
else:
    print("PoetMT already present")

# Clone CCPM directly into data/CCPM-master
if not os.path.exists("data/CCPM-master/train.jsonl"):
    !git clone https://github.com/THUNLP-AIPoet/CCPM.git data/CCPM-master
    print("CCPM contents:", os.listdir("data/CCPM-master"))
else:
    print("CCPM already present")


In [ ]:
# ── Cell 5: Build dataset ─────────────────────────────────────────────────
%cd /content/chinese_poetry_translation
!python build_dataset.py


In [ ]:
# ── Cell 6: Smoke test (1 epoch — verify pipeline before full run) ────────
%cd /content/chinese_poetry_translation
import json as _json
from collections import Counter
_tasks = Counter(_json.loads(l).get("task") for l in open("data/combined/train.jsonl") if l.strip())
print("Dataset task counts:", dict(_tasks))
assert _tasks.get("translation", 0) > 0, "No translation records found — re-run Cell 5"
!python pipelines/opus_mt/train_opus_mt.py --data_dir data/combined --output_dir /tmp/opus-mt-smoke --epochs 1 --precision bf16
print("Smoke test done — check output above before running Cell 7")


In [ ]:
# ── Cell 7: Full training (15 epochs, saved to Google Drive) ─────────────
%cd /content/chinese_poetry_translation
OUTPUT_DIR = "/content/drive/MyDrive/models/opus-mt-poetry"
!python pipelines/opus_mt/train_opus_mt.py --data_dir data/combined --output_dir {OUTPUT_DIR} --epochs 15 --precision bf16


In [ ]:
# ── Cell 8: Evaluate trained adapter on canonical 78-poem test set ───────
%cd /content/chinese_poetry_translation
OUTPUT_DIR  = "/content/drive/MyDrive/models/opus-mt-poetry"
ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"
EVAL_OUT    = f"{OUTPUT_DIR}/eval_results"
!python eval_e2_mt5.py --adapter_dir {ADAPTER_DIR} --base_model Helsinki-NLP/opus-mt-zh-en --no_task_prefix --flat_test --output_dir {EVAL_OUT}
